In [ ]:
# !pip install pyquaternion
# !pip install nuscenes-devkit efficientnet_pytorch==0.7.0

Возвращаемся к модели LSS.

Задание 1 - прогнать инференс на 1-10 примерах и написать визуализацию проекции BEV-сегментации на изображения. 

В ячейках ниже - повтор краткого содержания и концепции модели и основной код для прогона инференса.

Стоимость - 2 балла.

TLDR: Что такое Lift-Splat-Shot модель?

Это одна из базовых camera-only моделей, может быть включена в большую perception модель (например, BEV-Fusion), может быть использована самостоятельно.

Состоит из трёх этапов:

Lift (Подъём): На этом этапе изображения, полученные с камер, проецируются в трёхмерное пространство. Это делается с использованием информации о интринсиках камеры и глубине (или псевдоглубине), чтобы определить, где каждый пиксель изображения находится в трёхмерном пространстве.

Splat ("Разбрызгивание"): После проекции данных в 3D-пространство, они распределяются по трёхмерной сетке (Frutsum). Это позволяет создать плотное представление сцены, где каждый элемент сетки содержит информацию о своём окружении.

Shoot: На последнем этапе модель использует полученное трёхмерное представление для выполнения различных задач, таких как обнаружение объектов, оценка расстояний или прогнозирование движения. Это позволяет системе принимать решения на основе более полной информации о сцене.

In [ ]:
# код для скачивания данных

# !mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

# !wget https://www.nuscenes.org/data/v1.0-mini.tgz  # Download the nuScenes mini split.

# !tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

In [ ]:
# размер изображения

H=900
W=1600

# размеры на вход и на выход
resize_lim=(0.193, 0.225)
final_dim=(128, 352)
bot_pct_lim=(0.0, 0.22)
rot_lim=(-5.4, 5.4)
rand_flip=True

# параметры грида

xbound=[-50.0, 50.0, 0.5]
ybound=[-50.0, 50.0, 0.5]
zbound=[-10.0, 10.0, 20.0]
dbound=[4.0, 45.0, 1.0]

In [ ]:
grid_conf = {
        'xbound': xbound,
        'ybound': ybound,
        'zbound': zbound,
        'dbound': dbound,
    }
    
    
data_aug_conf = {
                    'resize_lim': resize_lim,
                    'final_dim': final_dim,
                    'rot_lim': rot_lim,
                    'H': H, 'W': W,
                    'rand_flip': rand_flip,
                    'bot_pct_lim': bot_pct_lim,
                    'cams': ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
                             'CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT'],
                    'Ncams': 5,
                }

In [ ]:
# download pretrained model
download_from_yadisk("https://disk.yandex.ru/d/m5xDFY8JnWsUhg", 
                               "model_pretrained.pt",
                               target_dir='.')

In [ ]:
from data import compile_data
from tools import (ego_to_cam, get_only_in_img_mask, denormalize_img,
                    SimpleLoss, get_val_info, add_ego, gen_dx_bx,
                    get_nusc_maps, plot_nusc_map, visualize_binmgs)
from models import compile_model

In [ ]:
# тест данных Nuscense
trainloader, valloader = compile_data("mini", "/data/sets/nuscenes", data_aug_conf=data_aug_conf,
                                          grid_conf=grid_conf, bsz=4, nworkers=1,
                                          parser_name='segmentationdata')

In [ ]:
device = torch.device(f'cuda:0')

model = compile_model(grid_conf, data_aug_conf, outC=1)
model.load_state_dict(torch.load('model_pretrained.pt'))
model.to(device)

loss_fn = SimpleLoss(1.0).cuda(0)

model.eval()

In [ ]:
# запуск одной итерации LSS на val

with torch.no_grad():
    for batch in valloader:
        allimgs, rots, trans, intrins, post_rots, post_trans, binimgs = batch
        preds = model(allimgs.to(device), rots.to(device),
                          trans.to(device), intrins.to(device), post_rots.to(device),
                          post_trans.to(device))
        binimgs = binimgs.to(device)
        break

In [ ]:
# место для решения

Задание 2: запустить обучение на Nuscense Mini, попробовать получить максимально хорошие метрики, близкие к тем, что заявлены в статье

https://arxiv.org/pdf/2008.05711

критерии выполнения: обученная модель, есть тензорборд с результатами, метрика не хуже 0.25 IOU по всем классам

стоимость - 2 балла

код обучения - https://github.com/yandexdataschool/sdc_course/blob/spring2025/seminar04-camera-only/src/train.py
запуск - https://github.com/yandexdataschool/sdc_course/blob/spring2025/seminar04-camera-only/src/run_train.sh

чтобы заработало, нужно добавить export NUSCENES_ROOT = /data/sets/nuscenes или любой другой путь, где у вас лежит nuscenes

можно править: lr, weight_decay

можно добавить понятных визуализаций для отладки модели, например, BEV плоскость и предсказания на BEV, которые мы получили выше, пример в следующей ячейке (+ 2 балла)

In [ ]:
scene_num = 3
bin_im = (binimgs[scene_num][0].detach().cpu().numpy()).astype(np.uint8) * 250
preds_im = (preds[scene_num][0].detach().cpu().numpy()).astype(np.uint8) * 250

In [ ]:
px.imshow(bin_im)

In [ ]:
px.imshow(preds_im)

Задание 3: добавить вариации IOU такие, что:
    
1. добавляют больше баллов ближайшим объектам на расстоянии r, заданном на вход метрике
2. учитывают перекрытие одним объектом другого на BEV-плоскости

необходимо прокинуть их в tensorboard и использовать в отладке и оценке обучения

стоимость - 4 балла